# 1장 2강: 유의수준·검정력·표본 크기의 트레이드오프와 실질적 유의성 — 실습문제

## 실습 목표

- 효과 크기와 유의수준이 같을 때 표본 크기에 따라 검정력이 어떻게 변하는지 설명할 수 있다.
- 목표 검정력과 유의수준을 이용해 필요한 표본 크기를 계산할 수 있다.
- 두 집단의 평균 차이에 대한 Cohen’s d를 계산하고 해석할 수 있다.
- p-value와 효과 크기, 실무 기준을 함께 사용하여 의사결정 근거를 작성할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- statsmodels
- `ames_housing.csv`

주요 컬럼은 다음과 같습니다.

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `CentralAir` | 중앙 냉방시설 유무(`Y`, `N`) |
| `KitchenQual` | 주방 품질(`Ex`, `Gd`, `TA`, `Fa`) |

> 모든 검정은 별도 지시가 없으면 유의수준 `α = 0.05`를 사용합니다.  
> Cohen’s d는 절댓값을 기준으로 약 0.2는 작은 효과, 0.5는 중간 효과, 0.8 이상은 큰 효과로 해석합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 전체 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.
import pandas as pd

df = pd.read_csv('ames_housing.csv')
print('데이터 크기: ',df.shape)

데이터 크기:  (1460, 10)


---

## 필수 1. 표본 크기와 검정력 비교

### 문제 1-1. 중앙 냉방시설 비교 연구에 필요한 표본 수 설계

#### 문제 설명

중앙 냉방시설이 있는 주택과 없는 주택의 판매가격을 비교하는 연구를 준비하고 있습니다. 사전 조사에서 표준화 효과 크기는 `0.3` 정도로 예상했습니다.

다음 조건에서 그룹당 표본 크기에 따른 검정력을 비교하고, 목표 검정력 0.8을 확보하기 위해 필요한 표본 수를 계산하세요.

- 예상 효과 크기: `0.3`
- 유의수준: `0.05`
- 양측검정
- 비교할 그룹당 표본 수: `30명`, `100명`, `300명`

#### 요구사항

1. `TTestIndPower()` 객체를 생성하세요.
2. 그룹당 표본 수가 30명, 100명, 300명일 때의 검정력을 각각 계산하세요.
3. 표본 크기별 검정력을 소수점 셋째 자리까지 출력하세요.
4. 유의수준 0.05, 목표 검정력 0.8, 효과 크기 0.3일 때 그룹당 필요한 표본 수를 계산하세요.
5. 계산된 표본 수는 소수점 이하를 올림하여 정수로 출력하세요.
6. 세 표본 크기 중 목표 검정력 0.8을 충족하는 경우를 확인하세요.

#### 해석 질문

**Q1.** 효과 크기와 유의수준이 같을 때 표본 크기가 커지면 검정력은 어떻게 변하나요?  
**Q2.** 검정력이 낮으면 실제 차이가 존재할 때 어떤 오류의 위험이 커지나요?  
**Q3.** 필요한 표본 수를 소수점 이하 올림으로 처리하는 이유는 무엇인가요?  
**Q4.** 표본 수를 무조건 크게 설정하는 것이 항상 최선인가요?

#### 제출 결과

- 표본 크기별 검정력
- 목표 검정력에 필요한 그룹당 표본 수
- 목표 검정력 충족 여부
- Q1~Q4 답변

In [2]:
# 필수 1 코드를 작성하세요.

from statsmodels.stats.power import TTestIndPower
import math

effect_size = 0.3   # 사전 조사에서 추정한 표준화 효과 크기 (Cohen's d)
alpha = 0.05         # 유의수준: "실제로 차이가 없는데 있다고 잘못 판단할 확률"의 허용 한도

analysis = TTestIndPower()  # 요구사항 1: 독립표본 t검정용 검정력 분석 객체 생성

# 요구사항 2~3: 그룹당 표본 수별 검정력 계산
sample_sizes = [30, 100, 300]
for n in sample_sizes:
    power = analysis.power(
        effect_size=effect_size,
        nobs1=n,          # 첫 번째 그룹(예: 냉방시설 있는 집)의 표본 수
        alpha=alpha,
        ratio=1.0,         # 두 그룹 표본 수 비율 (1.0 = 동일한 크기)
        alternative='two-sided'  # 양측검정
    )
    print(f'그룹당 표본 수 {n}명 -> 검정력: {power:.3f}')

# 요구사항 4~5: 목표 검정력 0.8을 위한 필요 표본 수 계산
required_n = analysis.solve_power(
    effect_size=effect_size, power=0.8, alpha=alpha, ratio=1.0, alternative='two-sided'
)
print('필요한 표본 수(올림):', math.ceil(required_n))

그룹당 표본 수 30명 -> 검정력: 0.208
그룹당 표본 수 100명 -> 검정력: 0.560
그룹당 표본 수 300명 -> 검정력: 0.956
필요한 표본 수(올림): 176


### 필수 1 답변 작성란
- 검정력이란?: (1-β) 차이가 존재할 때 귀무가설을 올바르게 기각할 확률
- 효과크기는 두 집단의 차이가 어느 정도 큰지를 나타내는 값이다.
- 효과 크기가 표본 크기가 커지면 일반적으로 검정력이 높아진다.
- 데이터의 변동성이 커지면 일반적으로 검정력이 낮아진다.

**Q1.** 효과 크기와 유의수준이 같을 때 표본 크기가 커지면 검정력은 어떻게 변하나요?  
<br>    - 표본 크기가 커지면 일반적으로 검정력이 높아진다.
<br>    - 표본이 많아질수록 실제 차이를 우연한 변동과 구분하기 쉬워짐

**Q2.** 검정력이 낮으면 실제 차이가 존재할 때 어떤 오류의 위험이 커지나요?  
<br>    - 실제 차이가 있는데도 귀무가설을 기가하지 못하는 제 2종 오류의 위험이 커짐
<br>    - 예를 들어 검정력이 0.8이라면 가정한 실제 차이를 놓칠 확률인 β는 ' 1- 0.8 '= 0.2이다.

**Q3.** 필요한 표본 수를 소수점 이하 올림으로 처리하는 이유는 무엇인가요?
<br>    - 사람이나 주택을 소수 단위로 모집할 수 없고
<br>    - 내림하면 계산된 목표 검정력을 충족하지 못할 수 있다.

**Q4.** 표본 수를 무조건 크게 설정하는 것이 항상 최선인가요?
<br>    - 아닙니다. 표본 확보에는 비용과 시간이 소요가 됨

---

## 필수 2. 통계적 유의성과 실질적 유의성 종합 판단

### 문제 2-1. 중앙 냉방시설 유무에 따른 판매가격 차이 분석

#### 문제 설명

한 부동산 회사는 중앙 냉방시설이 있는 주택과 없는 주택의 평균 판매가격 차이를 분석하려고 합니다. 회사는 두 집단의 평균 판매가격 차이가 **100,000달러 이상**이어야 냉방시설 설치 지원 사업을 검토할 실무적 가치가 있다고 정했습니다.

#### 요구사항

1. `CentralAir == "Y"`인 주택의 `SalePrice`를 `air_yes`에 저장하세요.
2. `CentralAir == "N"`인 주택의 `SalePrice`를 `air_no`에 저장하세요.
3. 두 집단의 표본 수와 평균 판매가격을 출력하세요.
4. `stats.ttest_ind()`로 두 집단 평균 차이에 대한 양측검정을 수행하세요.
5. 제공된 공식에 따라 Cohen’s d 계산 함수를 작성하고 효과 크기를 구하세요.
6. 평균 차이, p-value, Cohen’s d를 출력하세요.
7. 다음 기준으로 결과를 판단하세요.
   - `p-value ≤ 0.05`: 통계적으로 유의함
   - `|Cohen’s d| ≥ 0.8`: 큰 효과
   - `|평균 차이| ≥ 100000`: 회사 기준에서 실질적으로 유의함
8. 세 결과를 종합하여 사업 검토 여부에 대한 근거를 작성하세요.

#### Cohen’s d 공식

두 집단의 평균 차이를 합동 표준편차로 나누어 계산합니다.

$$
d = \frac{\bar{x}_1 - \bar{x}_2}{s_p}
$$

$$
s_p = \sqrt{\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1+n_2-2}}
$$

- $\bar{x}_1$, $\bar{x}_2$: 각 집단의 평균
- $s_1$, $s_2$: 각 집단의 표본 표준편차
- $n_1$, $n_2$: 각 집단의 표본 수
- $s_p$: 합동 표준편차

#### 해석 질문

**Q1.** p-value와 Cohen’s d는 각각 어떤 정보를 제공하나요?  
**Q2.** 두 집단의 판매가격 차이는 통계적으로 유의한가요?  
**Q3.** Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?  
**Q4.** 회사가 정한 100,000달러 기준에서 실질적으로 유의한가요?  
**Q5.** 통계적으로 유의하고 효과 크기가 크더라도 회사의 실무 기준을 충족하지 못할 수 있나요?

#### 제출 결과

- 집단별 표본 수와 평균
- 독립표본 t검정 결과
- 평균 차이와 Cohen’s d
- 통계적·효과크기·회사 기준 판단
- 최종 의사결정 근거
- Q1~Q5 답변

In [4]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('ames_housing.csv')

# 요구사항 1~2: 두 집단 분리
air_yes = df.loc[df['CentralAir'] == 'Y', 'SalePrice']
air_no = df.loc[df['CentralAir'] == 'N', 'SalePrice']

# 요구사항 3: 표본 수와 평균 출력
print(f'냉방 있음(Y): n={len(air_yes)}, 평균={air_yes.mean():,.0f}달러')
print(f'냉방 없음(N): n={len(air_no)}, 평균={air_no.mean():,.0f}달러')

# 요구사항 4: 독립표본 t검정 (양측검정)
t_stat, p_value = stats.ttest_ind(air_yes, air_no, equal_var=True, alternative='two-sided')

# 요구사항 5: Cohen's d 계산 함수
def cohens_d(x1, x2):
    n1, n2 = len(x1), len(x2)
    s1, s2 = x1.std(ddof=1), x2.std(ddof=1)   # ddof=1: 표본 표준편차(n-1로 나눔)
    sp = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))  # 합동 표준편차
    return (x1.mean() - x2.mean()) / sp

d = cohens_d(air_yes, air_no)
mean_diff = air_yes.mean() - air_no.mean()

# 요구사항 6: 결과 출력
print(f'평균 차이: {mean_diff:,.0f}달러')
print(f'p-value: {p_value:.3e}')
print(f"Cohen's d: {d:.3f}")

# ---------------------------------------------------------
# 요구사항 7: 세 가지 판단 기준 적용
# ---------------------------------------------------------
ALPHA = 0.05             # 유의수준 기준
D_THRESHOLD = 0.8        # 큰 효과 기준 (Cohen's d)
BUSINESS_THRESHOLD = 100000  # 회사가 정한 실무적 유의성 기준 (달러)

is_statistically_significant = p_value <= ALPHA
is_large_effect = abs(d) >= D_THRESHOLD
is_practically_significant = abs(mean_diff) >= BUSINESS_THRESHOLD

print('\n[판단 기준별 결과]')
print(f'1) 통계적 유의성 (p ≤ {ALPHA})       : {"충족 (유의함)" if is_statistically_significant else "미충족"}')
print(f'2) 큰 효과 (|d| ≥ {D_THRESHOLD})        : {"충족 (큰 효과)" if is_large_effect else "미충족"}')
print(f'3) 회사 실무 기준 (|차이| ≥ {BUSINESS_THRESHOLD:,}달러) : {"충족" if is_practically_significant else "미충족"}')

# ---------------------------------------------------------
# 요구사항 8: 종합 판단 (사업 검토 근거가 충분한지)
# ---------------------------------------------------------
print('\n[종합 판단]')
if is_statistically_significant and is_practically_significant:
    conclusion = "사업 검토 근거 충분: 통계적으로 유의하고, 회사 기준 금액 차이도 충족합니다."
elif is_statistically_significant and not is_practically_significant:
    conclusion = (
        "사업 검토 근거 부족: 통계적으로는 유의하지만(p값이 매우 작음), "
        f"실제 평균 차이({mean_diff:,.0f}달러)가 회사 기준({BUSINESS_THRESHOLD:,}달러)에 미치지 못합니다. "
        "통계적 유의성만으로는 사업성을 담보하지 못합니다."
    )
elif not is_statistically_significant and is_practically_significant:
    conclusion = (
        "사업 검토 근거 불확실: 평균 차이는 회사 기준을 넘지만, 통계적으로 유의하지 않아 "
        "표본에서 우연히 발생한 차이일 가능성을 배제할 수 없습니다. 추가 데이터 확인이 필요합니다."
    )
else:
    conclusion = "사업 검토 근거 없음: 통계적 유의성과 실무적 기준 모두 충족하지 못했습니다."

print(conclusion)
print(f'(참고: 효과 크기 판단 -> {"큰 효과" if is_large_effect else "큰 효과 아님"}, d={d:.3f})')

냉방 있음(Y): n=1365, 평균=186,187달러
냉방 없음(N): n=95, 평균=105,264달러
평균 차이: 80,923달러
p-value: 1.810e-22
Cohen's d: 1.052

[판단 기준별 결과]
1) 통계적 유의성 (p ≤ 0.05)       : 충족 (유의함)
2) 큰 효과 (|d| ≥ 0.8)        : 충족 (큰 효과)
3) 회사 실무 기준 (|차이| ≥ 100,000달러) : 미충족

[종합 판단]
사업 검토 근거 부족: 통계적으로는 유의하지만(p값이 매우 작음), 실제 평균 차이(80,923달러)가 회사 기준(100,000달러)에 미치지 못합니다. 통계적 유의성만으로는 사업성을 담보하지 못합니다.
(참고: 효과 크기 판단 -> 큰 효과, d=1.052)


### 필수 2 답변 작성란
**Q1.** p-value와 Cohen’s d는 각각 어떤 정보를 제공하나요?  
<BR> - P-value는 관측된 차이가 통게적으로 유의한지 판단하는 데 사용
<br> - Cohen's d는 두 집단 차이의 크기를 표준화하여 보여줌

**Q2.** 두 집단의 판매가격 차이는 통계적으로 유의한가요?  
<br> - 유의함 p-value가 0.05보다 작음 
<br> - 귀무가설: 냉방 시설 유무는 주택 가격 평균에 차이를 보이지 않는다. -> 기각

**Q3.** Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?  
<br> - Cohen's의 절대값이 1.05고 규정한 0.8보다 크기 때문에 큰 효과로 해석할 수 있다.

**Q4.** 회사가 정한 100,000달러 기준에서 실질적으로 유의한가요?  
<br> - 유의하지 않음, 80,900달러 정도였기 때문에 회사의 기대 미충족

**Q5.** 통계적으로 유의하고 효과 크기가 크더라도 회사의 실무 기준을 충족하지 못할 수 있나요?
<br> - 그렇다.
<br> - 통계적 유의성은 차이가 있다는 근거가 있는가?
<br> - 효과 크기는 자료의 흩어짐에 비해 차이가 큰가?
<br> - 회사 입장은 업무에서 요구하는 최소 금액 차이를 넘는가?

---

## 과제. 주방 품질에 따른 판매가격 차이 분석

### 문제 3-1. 주방 품질이 `Gd`인 집단과 `TA`인 집단 비교

#### 문제 설명

부동산 중개회사는 주방 품질이 `Gd`(Good)인 주택과 `TA`(Typical/Average)인 주택의 평균 판매가격을 비교하려고 합니다. 회사는 평균 판매가격 차이가 **50,000달러 이상**이면 마케팅에서 주방 품질의 차이를 강조할 실무적 가치가 있다고 판단합니다.

> 필수 2에서 학습한 검정과 효과 크기 계산 절차를 새로운 두 집단에 적용하는 과제입니다.

#### 요구사항

1. 주방 품질이 `Gd`인 집단과 `TA`인 집단의 `SalePrice`를 각각 준비하세요.
2. 두 집단의 표본 수와 평균 판매가격을 출력하세요.
3. 두 집단의 평균 차이에 대한 양측 독립표본 t검정을 수행하세요.
4. Cohen’s d를 계산하세요.
5. 평균 차이, p-value, Cohen’s d를 출력하세요.
6. 다음 기준에 따라 각각 판단하세요.
   - 통계적 유의성: `p-value ≤ 0.05`
   - 큰 효과: `|Cohen’s d| ≥ 0.8`
   - 실무적 유의성: `|평균 차이| ≥ 50000`
7. 세 가지 판단을 종합하여 주방 품질을 마케팅에서 강조할 근거가 있는지 결론을 작성하세요.

#### 해석 질문

**Q1.** 두 집단의 판매가격 차이는 통계적으로 유의한가요?  
**Q2.** Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?  
**Q3.** 회사가 정한 50,000달러 기준을 충족하나요?  
**Q4.** 최종적으로 주방 품질 차이를 마케팅에서 강조할 근거가 있다고 볼 수 있나요?

#### 제출 결과

- 집단별 표본 수와 평균
- t통계량과 p-value
- 평균 차이와 Cohen’s d
- 통계적·실질적 유의성 판단
- 최종 결론
- Q1~Q4 답변

In [9]:
# 과제 코드를 작성하세요.

import pandas as pd
import numpy as np
from scipy import stats

analysis = TTestIndPower()  # 독립표본 t검정용 검정력 분석 객체 생성


# 주방 품질이 `Gd`인 집단과 `TA`인 집단의 `SalePrice`를 각각 준비하세요.
gd_sample = df.loc[df['KitchenQual']=='Gd', 'SalePrice']
ta_sample = df.loc[df['KitchenQual']=='TA', 'SalePrice']


# 두 집단의 표본 수와 평균 판매가격을 출력하세요.
print(f'주방 품질 Gd: n={len(gd_sample)}, 평균={gd_sample.mean():,.0f}달러')
print(f'주방 품질 TA: n={len(ta_sample)}, 평균={ta_sample.mean():,.0f}달러')

# 두 집단의 평균 차이에 대한 양측 독립표본 t검정을 수행하세요.
t_stat, p_value = stats.ttest_ind(gd_sample, ta_sample, equal_var=True, alternative='two-sided')


# Cohen’s d를 계산하세요.
def cohens_d(x1, x2):
    n1, n2 = len(x1), len(x2)
    s1, s2 = x1.std(ddof=1), x2.std(ddof=1)   # ddof=1: 표본 표준편차(n-1로 나눔)
    sp = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))  # 합동 표준편차
    return (x1.mean() - x2.mean()) / sp

d = cohens_d(gd_sample, ta_sample)
mean_diff = gd_sample.mean() - ta_sample.mean()

# 평균 차이, p-value, Cohen’s d를 출력하세요.
print(f'평균 차이: {mean_diff:,.0f}달러')
print(f'p-value: {p_value:.6e}')
print(f'Cohen’s d: {d:.6f}')

# 기준에 따라 판단하기
print('\n[판단 기준별 결과]')
print(f'1) 통계적 유의성 (p ≤ 0.05): {"충족 (유의함)" if p_value <= 0.05 else "미충족"}')
print(f'2) 큰 효과 (|Cohen’s d| ≥ 0.8): {"충족 (큰 효과)" if abs(d) >= 0.8 else "미충족"}')
print(f'3) 실무적 유의성 (|평균 차이| ≥ 50000): {"충족" if abs(mean_diff) >= 50000 else "미충족"}') 

print('\n[종합 판단]')
if p_value <= 0.05 and abs(d) >= 0.8 and abs(mean_diff) >= 50000:
    conclusion = "사업 검토 근거 충분: 통계적으로 유의하고, 회사 기준 금액 차이도 충족한다."  
else:
    conclusion = "사업 검토 근거 부족: 통계적 유의성, 효과 크기, 실무적 유의성 중 하나 이상이 충족되지 않았다."
print(conclusion)   

주방 품질 Gd: n=586, 평균=212,116달러
주방 품질 TA: n=735, 평균=139,963달러
평균 차이: 72,154달러
p-value: 3.556775e-115
Cohen’s d: 1.399073

[판단 기준별 결과]
1) 통계적 유의성 (p ≤ 0.05): 충족 (유의함)
2) 큰 효과 (|Cohen’s d| ≥ 0.8): 충족 (큰 효과)
3) 실무적 유의성 (|평균 차이| ≥ 50000): 충족

[종합 판단]
사업 검토 근거 충분: 통계적으로 유의하고, 회사 기준 금액 차이도 충족한다.


### 과제 답변 작성란

- **Q1.** 두 집단의 판매가격 차이는 통계적으로 유의한가요?  
<br>-> p-value 값이 3.556775e-115로 유의 수준 0.05보다 작으므로 
귀무가설을 기각할 통계적 근거가 있다. 따라서 두 집단의 판매가격 차이가 있다는 대립가설을 지지할정도로 통계적으로 유의하다.

- **Q2.** Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?  
<br> -> Cohen's d를 기준으로 0.8을 넘으면 큰 효과라고 본다. 위 결과에서 Cohen's d는 1.399073으로 매우 큰 효과이다.


- **Q3.** 회사가 정한 50,000달러 기준을 충족하나요?  
<br>-> 평균 차이가 72,154달러로 회사가 정한 50,000달러 기준을 충족한다.

- **Q4.** 최종적으로 주방 품질 차이를 마케팅에서 강조할 근거가 있다고 볼 수 있나요?
<br>-> 통계적으로 유의하고 효과 크기가 충분하며 회사 입장에서 요구하는 최소 금액 차이를 넘기 때문에 최종적으로 마케팅에서 강조할 근거가 있다고 볼 수 있다.

---

## 실습 마무리

1. 검정력은 무엇이며 제2종 오류와 어떤 관계가 있나요?
2. 같은 효과 크기와 유의수준에서 표본 크기가 커지면 검정력은 어떻게 변하나요?
3. 목표 검정력이 높거나 발견하려는 효과가 작을수록 필요한 표본 수는 어떻게 변하나요?
4. p-value와 Cohen’s d를 함께 확인해야 하는 이유는 무엇인가요?
5. 통계적으로 유의한 결과가 반드시 실무적으로 중요한 결과를 의미하나요?